# ADCD Quickstart: Screened Coulomb — from Data to Discovery

**Asymptotic Dictionary Correction Discovery** — end-to-end walkthrough.

In this notebook you will:
1. Generate synthetic Screened Coulomb data (1% noise, historical r ≤ 4.0 m window)
2. Visualise the raw correction signal
3. Run the full ADCD pipeline (gate cascade → optimizer → BIC → verdict)
4. Understand why Time Dilation is correctly WITHHELD

**Expected runtime:** ~60 seconds on a laptop CPU (JAX CPU backend).

---
> **Ground truth is never shown to the pipeline.** ADCD only sees noisy observations
> and a taxonomy key. The correction expression is shown here for illustration only.

## 0. Setup

Run this notebook from the root of the `PhysicsPaper` repository.

If you haven't installed dependencies yet:
```bash
pip install -r requirements.txt
```

For Jupyter support:
```bash
pip install jupyter
jupyter notebook notebooks/01_quickstart.ipynb
```

In [ ]:
import sys, os

# Add src/ to path — works whether you run from repo root or notebooks/
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [os.path.join(repo_root, 'src'), 'src']:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt

print('Python:', sys.version.split()[0])
try:
    import jax; print('JAX:', jax.__version__)
except ImportError:
    print('WARNING: JAX not found — run: pip install jax jaxlib')
import sympy; print('SymPy:', sympy.__version__)

## 1. Load the Screened Coulomb Scenario

The **Screened Coulomb** scenario models electrostatic shielding:

| Component | Expression |
|---|---|
| Classical baseline | $V_{\rm cl}(r) = k_e\, q_1 q_2 / r$ |
| True correction | $\Delta = \theta_0 (e^{-r/\lambda_D} - 1)$ — Debye screening |
| Observation window | $r \leq 4.0$ m (historical low-signal regime) |
| Noise | 1% Gaussian multiplicative, seed 42 |

**The pipeline never sees `correction_expr`** — it only sees noisy observations, the classical baseline, dimensional variable definitions, and the taxonomy key.

In [ ]:
from adcd.anomaly_scenarios import get_all_scenarios

scenarios = {s.name: s for s in get_all_scenarios()}
scenario = scenarios['Screened Coulomb']

print(f'Scenario        : {scenario.name}')
print(f'Domain          : {scenario.domain}')
print(f'Classical expr  : {scenario.classical_expr}')
print(f'Correction class: {scenario.correction_class}')
print(f'Correction type : {scenario.correction_type}')
print()
print('Ground truth (hidden from pipeline below):')
print(f'  {scenario.correction_expr}')

In [ ]:
# Generate 200 synthetic observations with 1% Gaussian noise, seed 42
X, y_classical, y_obs, delta_true = scenario.generate_data(
    n_points=200,
    noise_level=0.01,   # 1% multiplicative Gaussian noise
    seed=42,
    domain_max=4.0,     # historical window r <= 4.0 m
)

print(f'Variables  : {list(X.keys())}')
print(f'N points   : {len(y_obs)}')
print(f'delta_true : min={delta_true.min():.4f}, max={delta_true.max():.4f}')
print(f'Correction magnitude: {np.abs(delta_true).mean()*100:.1f}% of classical')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

r = X['r']
delta_obs = y_obs / y_classical - 1
idx = np.argsort(r)

# Left: correction signal
ax = axes[0]
ax.scatter(r, delta_obs, s=12, alpha=0.5, color='gray', label='Observed Δ (1% noise)')
ax.plot(r[idx], delta_true[idx], 'b-', lw=2, label='True correction Δ')
ax.set_xlabel('r  [m]')
ax.set_ylabel('Correction Δ')
ax.set_title('Screened Coulomb — correction signal')
ax.legend()

# Right: raw observations vs classical
ax2 = axes[1]
ax2.scatter(r, y_obs, s=12, alpha=0.5, color='gray', label='Observed y')
ax2.plot(r[idx], y_classical[idx], 'b-', lw=2, label='Classical baseline')
ax2.set_xlabel('r  [m]')
ax2.set_ylabel('Potential  [arb. units]')
ax2.set_title('Screened Coulomb — raw observations')
ax2.legend()

plt.tight_layout()
plt.show()
print('Notice: the correction (left) is small but structured. ADCD must find this structure.')

## 2. Budget Disclosure (Step 1 of 4)

Before any fitting, ADCD announces the full search space size |C|.

For **Screened Coulomb**, the taxonomy key is `yukawa_debye_screening`:
- Allowed primitives: `D_exp`, `D_rat`
- Justified by: Klein-Gordon equation → V(r) ∝ e^{-mr}/r; Debye-Hückel → φ ∝ e^{-r/λ_D}/r
- The Buckingham-π engine auto-derives dimensionless ratio arguments
- **|C| = 60** total candidate expressions

In [ ]:
from adcd.quickfit import DOMAIN_TAXONOMY

taxonomy_key = 'yukawa_debye_screening'
allowed = DOMAIN_TAXONOMY[taxonomy_key]

print(f'Taxonomy key    : {taxonomy_key}')
print(f'Allowed prims   : {allowed}')
print()
print('Physical justification:')
print('  Klein-Gordon equation → V(r) ∝ e^{-mr}/r  [D_exp]')
print('  Debye-Hückel theory   → φ(r) ∝ e^{-r/λD}/r [D_exp + D_rat pole]')
print()
print('Budget |C| = 60 candidates (announced BEFORE any fitting)')

## 3. Full ADCD Pipeline

This calls the same entrypoint used to generate all paper results.
It runs all three scenarios; look for the **Screened Coulomb** block.

**Expected verdict: IDENTIFIABLE** with:
- Discovered expression: `θ₀(e^{-r/θ₄} − 1)`
- ΔBIC = 30.65 (far exceeds Kass-Raftery threshold of 10)
- NMSE = 2.77 × 10⁻⁴
- θ₄ ≈ 1.49 m recovering Debye length λ_D = 1.50 m

> Note: subscript **θ₄** (not θ₀) is machine-verifiable evidence of automated discovery —
> the Buckingham-π engine assigned this index by enumerating four charge-dimensional
> candidates before reaching the distance ratio r/θ.

In [ ]:
import subprocess

env = {**os.environ, 'PYTHONPATH': os.path.join(repo_root, 'src')}
result = subprocess.run(
    [sys.executable,
     os.path.join(repo_root, 'src', 'adcd', 'run_adcd_v3_validation_blind.py'),
     '--top-k', '5'],
    capture_output=True,
    text=True,
    cwd=repo_root,
    env=env,
)

output = result.stdout
# Show last 5000 chars to catch the summary table
print(output[-5000:] if len(output) > 5000 else output)

if result.returncode != 0:
    print('\n--- STDERR (last 2000 chars) ---')
    print(result.stderr[-2000:])

## 4. Parse the JSON Report

In [ ]:
import json

report_path = os.path.join(repo_root, 'run_outputs', 'adcd_v3_taxonomy_validation_report.json')

with open(report_path) as f:
    report = json.load(f)

# Print a clean summary regardless of report structure
entries = report if isinstance(report, list) else report.get('results', [report])

print(f'{'Scenario':<30} {'Verdict':<15} {'ΔBIC':>8}  {'NMSE':>10}')
print('-' * 68)
for entry in entries:
    name    = entry.get('scenario_name', entry.get('name', '?'))
    verdict = 'IDENTIFIABLE' if entry.get('all_passed', False) else 'WITHHELD'
    checks  = entry.get('checks', {})
    abla    = checks.get('ablation_control', {})
    dbic    = abla.get('delta_bic', float('nan'))
    pc      = checks.get('positive_control', {})
    nmse    = pc.get('nmse', float('nan'))
    print(f'{name:<30} {verdict:<15} {dbic:>8.2f}  {nmse:>10.4e}')

## 5. Understanding WITHHELD — Time Dilation

At v ≤ 0.3c, the Lorentz correction γ − 1 is only **4.8%** of the classical
prediction. Against 1% Gaussian noise, the signal-to-noise ratio is ≈ 4.8:1 —
insufficient for reliable isolation.

ADCD **correctly refuses to announce a discovery** rather than returning a
hallucinated formula. This is the intended, epistemically honest behavior:
a wider velocity window, better measurement precision, or more data points
would push ΔBIC above 10 with a passing positive control, yielding IDENTIFIABLE.

In [ ]:
scenario_td = scenarios['Time Dilation']
X_td, y_cl_td, y_obs_td, delta_true_td = scenario_td.generate_data(
    n_points=200, noise_level=0.01, seed=42, domain_max=0.3,
)

signal    = np.abs(delta_true_td).mean()
noise_lvl = 0.01

fig, ax = plt.subplots(figsize=(7, 4))
c = scenario_td.classical_constants.get('c', 1.0)
v = X_td['v'] / c
delta_obs_td = y_obs_td / y_cl_td - 1
idx_td = np.argsort(v)
ax.scatter(v, delta_obs_td, s=12, alpha=0.5, color='gray', label='Observed Δ (1% noise)')
ax.plot(v[idx_td], delta_true_td[idx_td], 'b-', lw=2, label='True Lorentz correction')
ax.set_xlabel('v/c')
ax.set_ylabel('Correction Δ')
ax.set_title('Time Dilation at v ≤ 0.3c — WITHHELD')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean correction magnitude : {signal:.4f}  ({signal*100:.1f}% of classical)')
print(f'Noise level               : {noise_lvl*100:.1f}%')
print(f'Effective SNR             : {signal/noise_lvl:.1f}:1')
print()
print('The 4.8% correction is buried under 1% noise at this observation window.')
print('ADCD issues WITHHELD — not a bug, but the epistemic safety gate working correctly.')

## Summary

| Step | What happened |
|---|---|
| Data generation | 200 noisy Screened Coulomb observations, seed 42 |
| Budget disclosure | 60 candidate expressions announced before any fitting |
| Gate cascade | Dimensional / ARC gates reject unphysical candidates |
| Optimization | JAX L-BFGS-B on surviving candidates (float64) |
| Verdict | **IDENTIFIABLE** — θ₀(e^{−r/θ₄}−1) recovered exactly |
| Key evidence | ΔBIC = 30.65, NMSE = 2.77 × 10⁻⁴, θ₄ ≈ 1.49 m ≈ λ_D |

The subscript θ₄ (not θ₀) is machine-verifiable evidence of automated discovery:
the Buckingham-π engine enumerated four charge-dimensional candidates before
reaching the distance ratio r/θ, assigning index 4 automatically.

---

**Next steps:**
- See `CONTRIBUTING.md` to add your own scenario
- Read `REPRODUCIBILITY.md` for number-by-number provenance
- Check `paper/main.pdf` for the full technical paper
- Run `bash reproduce_all.sh` (Linux/macOS) or `.\reproduce_all.ps1` (Windows)
  to verify all three scenarios and the determinism check